In [ ]:
import glob
import os
from pathlib import Path

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

from skimage.measure import label
from skimage.measure import label
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects
from skimage.morphology import remove_small_holes
from scipy.ndimage import gaussian_filter

import napari

import skimage
from scipy import ndimage as ndi
from skimage import measure, segmentation, filters
from skimage.measure import regionprops

from pathlib import Path

### Importing and loading

In [ ]:
# Provide path to input files

image_dir = Path(r'input_path')
image_path = os.path.join(image_dir,'*.tif') 
image_files = np.sort(glob.glob(image_path))

mask_cell_dir = Path(r'input_path')
mask_cell_path = os.path.join(mask_cell_dir,'*.tif') 
mask_cell_files = np.sort(glob.glob(mask_cell_path))

max_proj_dir = Path(r'input_path')
max_proj_path = os.path.join(max_proj_dir,'*.tif') 
max_proj_files = np.sort(glob.glob(max_proj_path))

print(len(image_files))
print(len(mask_cell_files))
print(len(max_proj_files))

In [ ]:
# Read images into list

images = []

for file in image_files:
    image = imread(file)
    images.append(image)

In [ ]:
plt.imshow(images[-1][0][0])

In [ ]:
# Read images into list

mask_cell = []
halo_images = []

for file in mask_cell_files:
    image = imread(file)
    mask_cell.append(image)

for file in max_proj_files:
    image = imread(file)
    halo_images.append(image)

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(mask_cell[-2])
ax[1].imshow(halo_images[-2])

### Nucleus segmentation

In [ ]:
# Creates a stack and calculates the global otsu threshold which will be used to threshold all images

halo_stack = np.hstack(halo_images)
halo_otsu_nucleus = threshold_otsu(halo_stack)*0.35 # Factor to adjust otsu
print(halo_otsu_nucleus)

In [ ]:
# Segments nuclei based on merged otsu threshold for substraction from whole cell masks

nucleus_binary = []
for img in halo_images:
    mask = (img >= halo_otsu_nucleus) # Binary mask
    nucleus_binary.append(mask)

for i in range(len(nucleus_binary)):
    mask_label = label(nucleus_binary[i].astype(np.int64)) # Mask needs to be converted to integers for following operations
    nucleus_binary[i] = gaussian_filter(nucleus_binary[i].astype(float), sigma=5) > 0.5 # Smoothen edges

In [ ]:
# Creates label images where small objects are removed for watershedding

nucleus_label = []

for img in halo_images:
    binary_mask = img >= halo_otsu_nucleus  # Create initial binary mask
    binary_mask = remove_small_objects(binary_mask, min_size=1000)
    smoothed_mask = gaussian_filter(binary_mask.astype(float), sigma=5) > 0.5  # Smooth edges, parameters can be fine-tuned
    smoothed_mask = remove_small_objects(smoothed_mask, min_size=1000)  # Remove small objects
    labeled_mask = label(smoothed_mask.astype(np.int64))  # Assign unique labels to smoothed mask
    labeled_mask = labeled_mask.astype(np.int32)
    nucleus_label.append(labeled_mask)

In [ ]:
print(len(nucleus_label))
print(nucleus_label[0].dtype)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (18,6))
ax[0].imshow(halo_images[-1])
ax[1].imshow(nucleus_binary[-1])
ax[2].imshow(nucleus_label[-1])

In [ ]:
print(np.unique(nucleus_binary[0]))
print(np.unique(nucleus_label[0]))

### Correcting nuclei

In [ ]:
# Create copies of cytoplasm masks for manipulation in Napari

mask_nucleus = nucleus_label.copy()
plt.imshow(mask_nucleus[-1])

In [ ]:
# Initializes Napari viewer

viewer = napari.Viewer()

In [ ]:
# Adds images to Napari viewer

for i, img in enumerate(images):
    layer_name = f'image_{i}'  # Dynamically name each layer
    viewer.add_image(img, name=layer_name)

In [ ]:
# Adds masks to Napari viewer

for i, img in enumerate(mask_nucleus):
    layer_name = f'segmentation_{i}'  # Dynamically name each layer
    labels_layer = viewer.add_labels(img, name=layer_name, opacity=0.2)

### Watershedding from nucleus to separate cell masks

In [ ]:
# Watershed using the nucleus as seed to separate the overlapping cell masks

mask_label=[]
distance_maps = []

for cell, nucleus in zip(mask_cell, mask_nucleus):
    
    # Smooth the distance transform of the cell mask
    distance_map = ndi.distance_transform_edt(cell)
    smoothed_distance_map = filters.gaussian(distance_map, sigma=1)
    distance_maps.append(smoothed_distance_map)

    # Use the nuclei labels as markers
    markers = measure.label(nucleus)

    # Apply the watershed algorithm
    labels = segmentation.watershed(-smoothed_distance_map, markers, mask=cell)
    mask_label.append(labels)

In [ ]:
# Plotting the results including the distance map
fig, ax = plt.subplots(1, 4, figsize=(20, 5))

ax[0].imshow(mask_cell[-2], cmap='gray')
ax[0].set_title('Cell Mask')

ax[1].imshow(mask_nucleus[-2], cmap='gray')
ax[1].set_title('Nucleus Mask')

ax[2].imshow(distance_maps[-2], cmap='gray')
ax[2].set_title('Distance Map')

ax[3].imshow(mask_label[-2])
ax[3].set_title('Separated Cell Labels')

for a in ax:
    a.axis('off')

plt.show()

In [ ]:
np.unique(mask_label[-1])

### Creating cytoplasm mask

In [ ]:
# To match intensity values (= labels) of ROIs in nucleus image to intensity values of ROIs in whole cell image

for nucleus_image, cell_image in zip(mask_nucleus, mask_label):
    # Create a dictionary to store coordinates and corresponding intensity values from mask_cell
    coordinates_intensity = {}

    # Store coordinates and intensity values from cell_image in the dictionary
    for cell_label in regionprops(cell_image):
        for coord in cell_label.coords:
            coordinates_intensity[tuple(coord)] = cell_image[coord[0], coord[1]]

    # Update nucleus_image based on the mapped coordinates and intensity values
    for nucleus_label in regionprops(nucleus_image):
        for coord in nucleus_label.coords:
            if tuple(coord) in coordinates_intensity:
                nucleus_image[coord[0], coord[1]] = coordinates_intensity[tuple(coord)]

In [ ]:
print(np.unique(mask_label[1]))
print(np.unique(mask_nucleus[1]))

In [ ]:
# Plot results of label matching

fig, ax = plt.subplots(1, 4, figsize = (16,4))
ax[0].imshow(mask_label[-1])
ax[1].imshow(mask_nucleus[-1])
ax[2].imshow(mask_label[-2])
ax[3].imshow(mask_nucleus[-2])

In [ ]:
# Substract nucleus mask from cell mask and store as binary cytoplasm mask

def subtract_nucleus_from_cell(cell_mask, nucleus_mask):
    """
    Subtract the nucleus mask from the cell mask and return the cytoplasm mask.
    """
    # Create a copy of the cell mask to modify
    cytoplasm_mask = cell_mask.copy()
    
    # Set the areas where nucleus label is present to 0 in the cell mask
    cytoplasm_mask[nucleus_mask > 0] = 0
    
    return cytoplasm_mask

# List to store cytoplasm masks
mask_cytoplasm = []

for cell, nucleus in zip(mask_label, mask_nucleus):
    # Subtract nucleus mask from cell mask
    cytoplasm_mask = subtract_nucleus_from_cell(cell, nucleus)
    
    mask_cytoplasm.append(cytoplasm_mask)

# Check results
for i, cytoplasm in enumerate(mask_cytoplasm):
    print(f"Unique values in cytoplasm mask {i}: {np.unique(cytoplasm)}")


In [ ]:
# Plot binary cytoplasm mask

plt.imshow(mask_cytoplasm[-1])

### Mask corrections with Napari for overlapping cells

Click pasteur pipette to pick a label. Use paint brush to draw a line to separate two labels. Fill entire area with according label with fill bucket option. To create background, select label 0 and use the paint brush to draw and bucket to fill up.

In [ ]:
# Create copies of cytoplasm masks for manipulation in Napari

mask_cytoplasm_copy = mask_cytoplasm.copy()
plt.imshow(mask_cytoplasm_copy[-1])

In [ ]:
# Initializes Napari viewer

#viewer = napari.Viewer()

In [ ]:
# Adds images to Napari viewer

for i, img in enumerate(images):
    layer_name = f'image_{i}'  # Dynamically name each layer
    viewer.add_image(img, name=layer_name)

In [ ]:
# Adds masks to Napari viewer

for i, img in enumerate(mask_cytoplasm_copy):
    layer_name = f'segmentation_{i}'  # Dynamically name each layer
    labels_layer = viewer.add_labels(img, name=layer_name, opacity=0.2)

#### Reload cytoplasm and nuclei masks for final label matching

In [ ]:
# Adds masks to Napari viewer

for i, img in enumerate(mask_cytoplasm_copy):
    layer_name = f'segmentation_{i}'  # Dynamically name each layer
    labels_layer = viewer.add_labels(img, name=layer_name, opacity=0.2)

In [ ]:
# Adds masks to Napari viewer

for i, img in enumerate(mask_nucleus):
    layer_name = f'segmentation_{i}'  # Dynamically name each layer
    labels_layer = viewer.add_labels(img, name=layer_name, opacity=1)

### Saving finished masks

Only saves masks that have more than one label, i.e. more than background.

When saving, then the "halo" part from the file name should be deleted, so that the masks have the same name as the original images except the _ROIs at the end (for spot detection).

After saving, images where no masks were created have to be removed from the Selected_for_analysis folder.

In [ ]:
# Provide path to output directory

mask_cytoplasm_dir = Path(r'output_path')
mask_nucleus_dir = Path(r'output_path')

In [ ]:
# Save mask images with removing last part of the file name

for img, tiff_file in zip(mask_cytoplasm_copy, max_proj_files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    
    # Replace spaces with underscores
    file_name = file_name.replace(' ', '_')
    
    # Remove the second-to-last part
    parts = file_name.split('_')
    if len(parts) > 2:  # Ensure there are enough parts to modify
        file_name = '_'.join(parts[:-2] + parts[-1:])
    
    print(file_name)
    
    # Check if the image has more than just the background (label 0)
    if len(np.unique(img)) > 1:  # This checks for more than one unique value
        # Define the output file path
        output_file = os.path.join(mask_cytoplasm_dir, file_name + '.tif')
        print(output_file)
        
        # Save the manipulated image as a TIFF
        tf.imwrite(output_file, img)
    else:
        print(f"Image {file_name} has only background (label 0), not saving.")


In [ ]:
# Save mask images with removing last part of the file name

for img, tiff_file in zip(mask_nucleus, max_proj_files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    
    # Replace spaces with underscores
    file_name = file_name.replace(' ', '_')
    
    print(file_name)
    
    # Check if the image has more than just the background (label 0)
    if len(np.unique(img)) > 1:  # This checks for more than one unique value
        # Define the output file path
        output_file = os.path.join(mask_nucleus_dir, file_name + '.tif')
        print(output_file)
        
        # Save the manipulated image as a TIFF
        tf.imwrite(output_file, img)
    else:
        print(f"Image {file_name} has only background (label 0), not saving.")
